# Can we predict next-month churn?
Target: `churn_next_month`. Data: monthly account snapshots (`data.csv`), dictionary in `columns.csv`.

Plan: inspect data → check for leakage → build baseline & models with honest validation (time-based, accounts never split across train/test) → report.

In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from sklearn.model_selection import GroupKFold
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.pipeline import make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import roc_auc_score, average_precision_score
pd.set_option('display.width',140)
df=pd.read_csv('data.csv'); df['snapshot_month']=pd.PeriodIndex(df.snapshot_month,freq='M')
print(df.shape, 'accounts:',df.account_ref.nunique()); print('missing:',df.isna().sum().sum(),' dup rows:',df.duplicated().sum())
df.describe(include='all').T

## Target and structure

In [ ]:
y='churn_next_month'
print('churn rate:',df[y].mean().round(4))
m=df.groupby('snapshot_month')[y].agg(['mean','size'])
fig,ax=plt.subplots(1,2,figsize=(12,3.5))
m['mean'].plot(ax=ax[0],marker='o',title='Churn rate by snapshot month'); m['size'].plot.bar(ax=ax[1],title='Snapshots per month')
plt.tight_layout(); plt.show()
print(df.groupby('account_ref').size().describe())

Churn is far from stationary: ~3-5% in spring 2024 vs >50% around Dec 2024–Jan 2025. Accounts also appear many times (up to 23 snapshots), so random row splits would leak account identity between train and test.

## Leakage check: `lifetime_months`
The dictionary says it is the *total* months an account was active, taken from a CRM export on 2026-01-05, i.e. it is measured **after** the snapshot. Compare it with `tenure_months`:

In [ ]:
df['gap']=df.lifetime_months-df.tenure_months
print(df.groupby(y).gap.describe())
print('churners with lifetime==tenure:',(df[df[y]==1].gap==0).mean())

`lifetime_months` equals `tenure_months` for **every** churner and exceeds it for most non-churners — it encodes the outcome (an account's end date is only known once it has ended). It would not be available at prediction time, so **it is excluded**. The next cell shows how misleading it would be.

In [ ]:
num=['seats','monthly_fee','tenure_months','usage_hours','support_tickets']; cat=['plan']
def pre(nums): return ColumnTransformer([('n',StandardScaler(),nums),('c',OneHotEncoder(handle_unknown='ignore'),cat)])
def hgb(): return HistGradientBoostingClassifier(max_depth=4,learning_rate=0.05,max_iter=300,random_state=0)
def evaluate(model,feats,tr,te):
    model.fit(tr[feats],tr[y]); p=model.predict_proba(te[feats])[:,1]
    return roc_auc_score(te[y],p),average_precision_score(te[y],p),p
# forward time split: train on early months, test on the latest snapshot months.
# The label for month t is cancellation in t+1, so leave one month unused between train and test.
cut=pd.Period('2025-06','M'); tr,te=df[df.snapshot_month<cut-1],df[df.snapshot_month>=cut]
print(len(tr),len(te),'test base rate',te[y].mean().round(3))
leak=evaluate(make_pipeline(pre(num+['lifetime_months']),LogisticRegression(max_iter=1000)),num+['lifetime_months']+cat,tr,te)
print('WITH leaky lifetime_months  AUC=%.3f  AP=%.3f  (not trustworthy)'%leak[:2])

## Experiments (leak-free features)
Features: plan, seats, monthly_fee, tenure_months, usage_hours, support_tickets. Two validations:
1. **Grouped K-fold** by account (accounts unseen in training).
2. **Forward time split**: train ≤ 2025-04, test 2025-06 → 2025-11 (closest to real deployment).

Baseline = predicting the base rate (AUC 0.5, average precision ≈ prevalence).

In [ ]:
feats=num+cat
models={'logreg':make_pipeline(pre(num),LogisticRegression(max_iter=1000)),
        'grad_boost':make_pipeline(pre(num),hgb())}
rows=[]
for name,mdl in models.items():
    a,p=[],[]
    for tri,tei in GroupKFold(5).split(df,groups=df.account_ref):
        r=evaluate(mdl,feats,df.iloc[tri],df.iloc[tei]); a.append(r[0]); p.append(r[1])
    rows.append((name,'grouped 5-fold',np.mean(a),np.std(a),np.mean(p)))
    r=evaluate(mdl,feats,tr,te); rows.append((name,'forward time split',r[0],np.nan,r[1]))
res=pd.DataFrame(rows,columns=['model','validation','AUC','AUC_sd','avg_precision'])
print('prevalence: overall %.3f, time-test %.3f'%(df[y].mean(),te[y].mean())); res.round(3)

## What drives churn?

In [ ]:
fig,ax=plt.subplots(1,3,figsize=(15,3.5))
for a,c in zip(ax,['tenure_months','usage_hours','support_tickets']):
    q=pd.qcut(df[c],10,duplicates='drop') if df[c].nunique()>8 else df[c]
    df.groupby(q,observed=True)[y].mean().plot(ax=a,marker='o',title=f'churn rate vs {c}'); a.tick_params(axis='x',rotation=45)
plt.tight_layout(); plt.show()
print(df.groupby('plan')[y].agg(['mean','size']))

In [ ]:
from sklearn.inspection import permutation_importance
mdl=models['grad_boost'].fit(tr[feats],tr[y])
pi=permutation_importance(mdl,te[feats],te[y],scoring='roc_auc',n_repeats=10,random_state=0)
pd.Series(pi.importances_mean,index=feats).sort_values().plot.barh(title='Permutation importance (test AUC drop)'); plt.show()

## Does calendar time matter?
Churn swings strongly by month, which cannot be extrapolated from 23 months. Test whether month-of-year adds anything, and how performance varies by test month.

In [ ]:
df['moy']=df.snapshot_month.dt.month; tr,te=df[df.snapshot_month<cut-1],df[df.snapshot_month>=cut]
f2=feats+['moy']
m2=make_pipeline(ColumnTransformer([('n','passthrough',num+['moy']),('c',OneHotEncoder(handle_unknown='ignore'),cat)]),hgb())
r=evaluate(m2,f2,tr,te); print('grad_boost + month-of-year (time split): AUC=%.3f AP=%.3f'%r[:2])
p=evaluate(models['grad_boost'],feats,tr,te)[2]
t=te.assign(p=p)
rows=[(str(k),len(g),g[y].mean(),roc_auc_score(g[y],g.p) if g[y].nunique()>1 else np.nan) for k,g in t.groupby('snapshot_month')]
pd.DataFrame(rows,columns=['month','n','churn_rate','AUC']).round(3)

In [ ]:
# lift: churn rate by predicted-risk decile in the time-split test set (10 = highest risk)
t['decile']=pd.qcut(t.p.rank(method='first'),10,labels=False)+1
t.groupby('decile')[y].agg(['mean','size']).round(3).T

## Conclusions
**Answer: partly.** With leak-free features, churn is predictable only modestly (AUC ≈ 0.67–0.69, vs 0.5 for chance) — useful for ranking/prioritising accounts, not for confident individual predictions.

*(Figures below come from a run of this notebook's code; re-run in Jupyter to regenerate outputs — the cells in this file were saved without outputs.)*

1. **`lifetime_months` is target leakage and was excluded.** It equals `tenure_months` for 100% of churners (the CRM export records the final lifetime, known only after cancellation). Including it gives AUC = 1.00, which would be an illusory result. Do not use it in production.
2. **Honest performance:** grouped 5-fold AUC 0.665 (logreg) / 0.680 (gradient boosting), sd ≈ 0.02; forward time split (train ≤ Apr 2025, test Jun–Nov 2025) AUC ≈ 0.69 for both. Average precision on the time test is ≈ 0.20–0.21 vs 0.093 prevalence (≈ 2.2× lift). Gradient boosting gives no real gain over logistic regression, so the signal is mostly simple and roughly monotonic.
3. **Lift:** in the time-split test, the top predicted-risk decile churns at ≈ 23% vs ≈ 3–5% in the lowest deciles — a ~5× spread between the best and worst groups.
4. **Drivers:** support tickets (strongest; churn 10% with 0 tickets → 39% with 3+), then low usage (23% in the lowest usage quintile → 9% in the highest), then seats/fee. `plan` and `tenure_months` add essentially nothing.
5. **Caveats.**
   - Base churn rate swings from ~3% to >50% across calendar months (late 2024–Jan 2025 spike). Features explain little of this; model scores are better for *ranking* accounts within a month than for forecasting the *level* of churn. Adding month-of-year did not help (AUC 0.673).
   - Calibration was not tuned; recalibrate on recent data before using scores as probabilities.
   - Only one split of 23 months was tested; the time-test period is calmer than the spike period, so performance during a churn wave is unverified.
   - Next steps: account-level history features (usage trend, ticket change vs prior month), and investigate what caused the Dec–Jan spike.